# 05 — Training
Student Success Analytics Platform

In [1]:
import sys
print(sys.executable)

C:\Users\lanre\anaconda3\python.exe


In [2]:
import sys
!{sys.executable} -m pip install mlflow

In [3]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import json
import pandas as pd
import mlflow

from src.training import (
    train_linear_regression, train_xgb_regressor,
    train_logistic_regression, train_xgb_classifier,
    evaluate_regression, evaluate_classification, log_run
)

mlflow.set_experiment("student-success-platform")
print("libraries loaded, mlflow experiment set")

2026/06/27 23:25:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/27 23:25:33 INFO mlflow.store.db.utils: Updating database tables
2026/06/27 23:25:37 INFO mlflow.tracking.fluent: Experiment with name 'student-success-platform' does not exist. Creating a new experiment.


libraries loaded, mlflow experiment set


In [4]:
with open("../data/processed/selected_features.json") as f:
    selected_features = json.load(f)

X_train_selected = pd.read_csv("../data/processed/X_train_selected.csv")
X_test_selected = pd.read_csv("../data/processed/X_test_selected.csv")
y_reg_train = pd.read_csv("../data/processed/y_reg_train.csv").squeeze()
y_reg_test = pd.read_csv("../data/processed/y_reg_test.csv").squeeze()
y_clf_test = pd.read_csv("../data/processed/y_clf_test.csv").squeeze()

# X_train_resampled was saved in notebook 3, BEFORE notebook 4 dropped wellness_score/distraction_hours —
# it still has 58 columns. Filter it down to the same 56 selected features before training the classifier,
# or the column set used at train time won't match what's used at inference/eval time.
X_train_resampled_raw = pd.read_csv("../data/processed/X_train_resampled.csv")
y_clf_train_resampled = pd.read_csv("../data/processed/y_clf_train_resampled.csv").squeeze()
X_train_resampled = X_train_resampled_raw[selected_features]

print(f"selected features: {len(selected_features)}")
print(f"regression train/test: {X_train_selected.shape} / {X_test_selected.shape}")
print(f"classification train (resampled, filtered): {X_train_resampled.shape}")

selected features: 56
regression train/test: (64000, 56) / (16000, 56)
classification train (resampled, filtered): (125468, 56)


## Regression — exam_score
`previous_gpa` alone carried 87.5% of feature importance in notebook 4's diagnostic pass. Training it as a standalone naive baseline first, so the full models' improvement (or lack of it) over a single column is reported honestly rather than just citing R² in isolation.

In [5]:
naive_model = train_linear_regression(X_train_selected[["previous_gpa"]], y_reg_train)
naive_metrics = evaluate_regression(naive_model, X_test_selected[["previous_gpa"]], y_reg_test)
print("naive (previous_gpa only):", naive_metrics)

naive (previous_gpa only): {'rmse': 4.155319133616997, 'mae': 3.1994198635256463, 'r2': 0.869651369527216}


In [6]:
linear_model = train_linear_regression(X_train_selected, y_reg_train)
linear_metrics = evaluate_regression(linear_model, X_test_selected, y_reg_test)
print("linear regression (all features):", linear_metrics)

linear regression (all features): {'rmse': 4.1571788445914635, 'mae': 3.2005780996408766, 'r2': 0.8695346684911651}


In [7]:
xgb_reg_model = train_xgb_regressor(X_train_selected, y_reg_train)
xgb_reg_metrics = evaluate_regression(xgb_reg_model, X_test_selected, y_reg_test)
print("xgboost regressor (all features):", xgb_reg_metrics)

xgboost regressor (all features): {'rmse': 4.301136026466039, 'mae': 3.3306329250335693, 'r2': 0.8603425621986389}


In [8]:
reg_comparison = pd.DataFrame({
    "naive (previous_gpa only)": naive_metrics,
    "linear regression": linear_metrics,
    "xgboost": xgb_reg_metrics,
}).T
reg_comparison

,rmse,mae,r2
naive (previous_gpa only),4.155319,3.199420,0.869651
linear regression,4.157179,3.200578,0.869535
xgboost,4.301136,3.330633,0.860343


## Classification — dropout_risk
Trained on the SMOTE-resampled training set (balanced 50/50), evaluated exclusively on the untouched, naturally-imbalanced test set (~2% positive) — evaluating on resampled data would give a falsely optimistic read.

In [9]:
logistic_model = train_logistic_regression(X_train_resampled, y_clf_train_resampled)
logistic_metrics = evaluate_classification(logistic_model, X_test_selected, y_clf_test)
print("logistic regression:")
for k, v in logistic_metrics.items():
    print(f"  {k}: {v}")

logistic regression:
  accuracy: 0.9824375
  precision: 0.5298126064735945
  recall: 0.9841772151898734
  f1: 0.6888150609080842
  roc_auc: 0.9965279459192469
  confusion_matrix: [[15408, 276], [5, 311]]


In [10]:
xgb_clf_model = train_xgb_classifier(X_train_resampled, y_clf_train_resampled)
xgb_clf_metrics = evaluate_classification(xgb_clf_model, X_test_selected, y_clf_test)
print("xgboost classifier:")
for k, v in xgb_clf_metrics.items():
    print(f"  {k}: {v}")

xgboost classifier:
  accuracy: 1.0
  precision: 1.0
  recall: 1.0
  f1: 1.0
  roc_auc: 1.0
  confusion_matrix: [[15684, 0], [0, 316]]


In [11]:
clf_comparison = pd.DataFrame({
    "logistic regression": {k: v for k, v in logistic_metrics.items() if k != "confusion_matrix"},
    "xgboost": {k: v for k, v in xgb_clf_metrics.items() if k != "confusion_matrix"},
}).T
clf_comparison

,accuracy,precision,recall,f1,roc_auc
logistic regression,0.982437,0.529813,0.984177,0.688815,0.996528
xgboost,1.000000,1.000000,1.000000,1.000000,1.000000


## MLflow logging
All four models logged as separate runs under one experiment, with params, metrics, and the model artifact itself.

In [12]:
log_run("naive_previous_gpa_only", {"features": "previous_gpa"}, naive_metrics, naive_model, model_type="sklearn")
log_run("linear_regression_full", {"features": len(selected_features)}, linear_metrics, linear_model, model_type="sklearn")
log_run("xgboost_regressor", {"n_estimators": 200, "features": len(selected_features)}, xgb_reg_metrics, xgb_reg_model, model_type="xgboost")
log_run("logistic_regression_clf", {"features": len(selected_features), "resampled": True}, logistic_metrics, logistic_model, model_type="sklearn")
log_run("xgboost_classifier", {"n_estimators": 200, "features": len(selected_features), "resampled": True}, xgb_clf_metrics, xgb_clf_model, model_type="xgboost")

print("5 runs logged to mlflow experiment 'student-success-platform'")

2026/06/27 23:25:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/27 23:26:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/27 23:26:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/27 23:26:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/27 23:26:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


5 runs logged to mlflow experiment 'student-success-platform'


In [13]:
import joblib
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(xgb_reg_model, "../models/exam_score_model.joblib")
joblib.dump(xgb_clf_model, "../models/dropout_risk_model.joblib")

print("best models saved to ../models/ for use by the API")

best models saved to ../models/ for use by the API


In [14]:
print("=" * 50)
print("  TRAINING SUMMARY")
print("=" * 50)
print("  regression (exam_score):")
print(reg_comparison.round(3).to_string())
print()
print("  classification (dropout_risk):")
print(clf_comparison.round(3).to_string())
print("=" * 50)
print()
print("next: 06_explainability.ipynb")

  TRAINING SUMMARY
  regression (exam_score):
                            rmse    mae    r2
naive (previous_gpa only)  4.155  3.199  0.87
linear regression          4.157  3.201  0.87
xgboost                    4.301  3.331  0.86

  classification (dropout_risk):
                     accuracy  precision  recall     f1  roc_auc
logistic regression     0.982       0.53   0.984  0.689    0.997
xgboost                 1.000       1.00   1.000  1.000    1.000

next: 06_explainability.ipynb


In [15]:
from sklearn.tree import DecisionTreeClassifier

shallow_tree = DecisionTreeClassifier(max_depth=2, random_state=42)
shallow_tree.fit(X_train_resampled, y_clf_train_resampled)
shallow_metrics = evaluate_classification(shallow_tree, X_test_selected, y_clf_test)
print(shallow_metrics)

{'accuracy': 1.0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'roc_auc': 1.0, 'confusion_matrix': [[15684, 0], [0, 316]]}


In [16]:
from sklearn.tree import export_text
print(export_text(shallow_tree, feature_names=list(X_train_resampled.columns)))

|--- stress_level <= 1.56
|   |--- class: 0
|--- stress_level >  1.56
|   |--- motivation_level <= -0.69
|   |   |--- class: 1
|   |--- motivation_level >  -0.69
|   |   |--- class: 0

